# What does `explain` do? A walk-through on one order

This notebook runs the whole Precedent idea on **one** sales order and prints every step, so you can see what goes in and what comes out.

It takes about 5 minutes after the data has loaded. Same setup as before: GPU T4, Internet on, secrets `TABPFN_TOKEN` and `HF_TOKEN` attached. You can run it cell by cell in the editor; no need to commit.

**The steps**

1. Load the past orders and fit the full model.
2. Pick one new order and look at the model's answer.
3. `explain`: find the past orders the answer rests on.
4. Read the result: the new order next to its precedents.
5. `verify`: remove the precedents from the full model and see whether the answer moves.
6. The final result, in the form the tool would return it.

In [1]:
!pip install -q -U tabpfn datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 12.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 21.9 MB/s eta 0:00:00


In [2]:
import os, time, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

TABPFN_TOKEN, HF_TOKEN = get_secret("TABPFN_TOKEN"), get_secret("HF_TOKEN")
assert TABPFN_TOKEN, "Attach the TABPFN_TOKEN secret to this notebook."
os.environ["TABPFN_TOKEN"] = TABPFN_TOKEN
os.environ["TABPFN_NO_BROWSER"] = "1"
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN

import torch, tabpfn
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
print("tabpfn", tabpfn.__version__, "| GPU:", torch.cuda.is_available())

def make_model(fast=True, **overrides):
    version = ModelVersion.V3_5_FAST if fast else ModelVersion.V3_5
    return TabPFNClassifier.create_default_for_version(version, **overrides)

tabpfn 9.1.0 | GPU: True


## Settings

In [4]:
TARGET       = "CUSTOMERPAYMENTTERMS"
N_CONTEXT    = 2000     # solved examples the model may look at
N_TEST       = 1000     # orders used for the accuracy comparison
N_EXPLAIN    = 1        # this notebook explains one order at a time
K            = 60       # candidates: the 60 past orders most similar to the new one
M            = 180      # hide-and-ask rounds (3 x K)
TOP_K        = 10       # rows removed in each removal test
N_RANDOM     = 3        # random-removal repeats per order (full model)
N_FRESH      = 20       # fresh subsets for the surrogate check
N_EST        = 1        # TabPFN ensemble members, used for BOTH the small and the full model
MAX_HOURS    = 5.0      # time budget for the loop
SEED         = 0
RESULTS_CSV  = "precedent_full_results.csv"

SALT_TARGETS = ["SALESOFFICE", "SALESGROUP", "CUSTOMERPAYMENTTERMS", "SHIPPINGCONDITION",
                "PLANT", "SHIPPINGPOINT", "HEADERINCOTERMSCLASSIFICATION", "ITEMINCOTERMSCLASSIFICATION"]
ID_COLUMNS   = ["SALESDOCUMENT", "SALESDOCUMENTITEM"]   # identifiers, not business features
CUSTOMER_COL = "SOLDTOPARTY"
SAMPLE_ROWS  = 300_000

## Step 1. Load the past orders and fit the full model

In [5]:
def load_frame():
    from datasets import load_dataset
    ds = load_dataset("SAP/SALT", "joined_table", split="train", token=HF_TOKEN)
    return ds.to_pandas()

df = load_frame()
print("Loaded:", df.shape)
df = df.sample(n=min(SAMPLE_ROWS, len(df)), random_state=SEED)

def is_text(col):
    t = col.dtype
    return (pd.api.types.is_object_dtype(t) or pd.api.types.is_string_dtype(t)
            or isinstance(t, pd.CategoricalDtype))

# clean the target: drop missing and blank codes
df[TARGET] = df[TARGET].astype(object)
df = df[df[TARGET].notna()]
df[TARGET] = df[TARGET].astype(str).str.strip()
n_before = len(df)
df = df[df[TARGET] != ""]
print(f"Removed {n_before - len(df)} rows with a blank target")

# drop the other SALT targets (leakage), identifiers, and leftover index columns
drop = [c for c in df.columns if (c in SALT_TARGETS and c != TARGET) or c in ID_COLUMNS or c.startswith("__")]
df = df.drop(columns=drop)
print("Dropped columns:", drop)

top = df[TARGET].value_counts().index[:10]
df = df[df[TARGET].isin(top)]
print(df[TARGET].value_counts(normalize=True).round(3))

README.md:   0%|          | 0.00/6.70k [00:00<?, ?B/s]

JoinedTables_train.parquet: reconstructing file:   0%|          |  0.00B / 42.8MB            

JoinedTables_train.parquet: downloading bytes:           |  0.00B            

JoinedTables_test.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

JoinedTables_test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1916685 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/402855 [00:00<?, ? examples/s]

Loaded: (1916685, 29)
Removed 3426 rows with a blank target
Dropped columns: ['SALESDOCUMENT', 'SALESOFFICE', 'SALESGROUP', 'SHIPPINGCONDITION', 'HEADERINCOTERMSCLASSIFICATION', 'SALESDOCUMENTITEM', 'PLANT', 'SHIPPINGPOINT', 'ITEMINCOTERMSCLASSIFICATION', '__index_level_0__']
CUSTOMERPAYMENTTERMS
32    0.297
03    0.239
54    0.149
33    0.107
00    0.067
95    0.036
42    0.030
01    0.028
96    0.026
70    0.020
Name: proportion, dtype: float64


In [6]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
rng = np.random.default_rng(SEED)

date_col = next((c for c in df.columns if "DATE" in c.upper()), None)
if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.sort_values(date_col)
else:
    df = df.sample(frac=1.0, random_state=SEED)
cut = int(len(df) * 0.8)
past, future = df.iloc[:cut], df.iloc[cut:]
print("Split on:", date_col or "random", "| past:", len(past), "| future:", len(future))

ctx = past.sample(n=min(N_CONTEXT, len(past)), random_state=SEED)
tst = future[future[TARGET].isin(ctx[TARGET].unique())]
tst = tst.sample(n=min(N_TEST, len(tst)), random_state=SEED)

feat = [c for c in df.columns if c != TARGET]
dropped = []
for c in list(feat):
    nun = ctx[c].nunique(dropna=True)
    if nun <= 1 or (is_text(ctx[c]) and nun > 0.95 * len(ctx)):
        feat.remove(c); dropped.append(c)
print("Dropped (constant or all-unique):", dropped)
print("Features used:", feat)

def prep(frame):
    out = frame[feat].copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = (out[c] - pd.Timestamp("1970-01-01")).dt.days
        elif is_text(out[c]):
            out[c] = out[c].astype(object).where(out[c].notna(), "missing").astype(str).astype(object)
    return out.reset_index(drop=True)

X_ctx, X_tst = prep(ctx), prep(tst)
y_ctx, y_tst = ctx[TARGET].to_numpy(), tst[TARGET].to_numpy()

cat_cols = [c for c in feat if is_text(X_ctx[c])]
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
E_ctx, E_tst = X_ctx.copy(), X_tst.copy()
if cat_cols:
    E_ctx[cat_cols] = enc.fit_transform(X_ctx[cat_cols])
    E_tst[cat_cols] = enc.transform(X_tst[cat_cols])
E_ctx = E_ctx.astype(float).fillna(-1); E_tst = E_tst.astype(float).fillna(-1)
print("Context:", X_ctx.shape, "| Test:", X_tst.shape, "| Classes:", len(np.unique(y_ctx)))

Split on: CREATIONDATE | past: 193956 | future: 48490
Dropped (constant or all-unique): ['ORGANIZATIONDIVISION']
Features used: ['SALESDOCUMENTTYPE', 'SALESORGANIZATION', 'DISTRIBUTIONCHANNEL', 'BILLINGCOMPANYCODE', 'TRANSACTIONCURRENCY', 'CREATIONDATE', 'CREATIONTIME', 'SALESDOCUMENTITEMCATEGORY', 'PRODUCT', 'SOLDTOPARTY', 'SHIPTOPARTY', 'BILLTOPARTY', 'PAYERPARTY', 'ADDRESSID_SOLDTOPARTY', 'ADDRESSID_BILLTOPARTY', 'ADDRESSID_PAYERPARTY', 'ADDRESSID_SHIPTOPARTY']
Context: (2000, 17) | Test: (1000, 17) | Classes: 10


In [7]:
def new_model():
    return make_model(fast=True, n_estimators=N_EST, random_state=SEED)

full_model = new_model().fit(X_ctx, y_ctx)          # TabPFN reads all 2,000 past orders
proba_full = full_model.predict_proba(X_tst)        # and predicts every new order
pred_full  = full_model.classes_[proba_full.argmax(1)]
conf_full  = proba_full.max(1)
print("Past orders (context):", len(X_ctx), "| new orders:", len(X_tst))
print("Share of new orders predicted correctly:", round(float((pred_full == y_tst).mean()), 3))

tabpfn-v3.5-fast-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  334MB            

tabpfn-v3.5-fast-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

Past orders (context): 2000 | new orders: 1000
Share of new orders predicted correctly: 0.646


## Step 2. Pick one new order

By default the notebook picks an order where the model is confident **and** the same customer appears among the past orders, because run 3 showed those are the cases where explanations tend to work. To look at a different order, set `ORDER` to any number from 0 to 999.

In [8]:
ORDER = None     # None = pick automatically; or set a number, e.g. ORDER = 27

if ORDER is None:
    known_customers = set(X_ctx[CUSTOMER_COL]) if CUSTOMER_COL in X_ctx.columns else set()
    good = [j for j in range(len(X_tst))
            if 0.90 <= conf_full[j] <= 0.999 and X_tst[CUSTOMER_COL].iloc[j] in known_customers]
    ORDER = good[0] if good else int(np.argmax(conf_full))

x_row = X_tst.iloc[[ORDER]]
answer, p_before = pred_full[ORDER], float(conf_full[ORDER])
print(f"New order number {ORDER}")
print(f"Model's answer : payment terms '{answer}'  ({p_before:.0%} sure)")
print(f"True answer    : payment terms '{y_tst[ORDER]}'")
print("\nThe new order's fields:")
x_row.T.rename(columns={x_row.index[0]: "new order"})

New order number 7
Model's answer : payment terms '03'  (97% sure)
True answer    : payment terms '03'

The new order's fields:


,new order
SALESDOCUMENTTYPE,TA
SALESORGANIZATION,0010
DISTRIBUTIONCHANNEL,10
BILLINGCOMPANYCODE,D011
TRANSACTIONCURRENCY,EUR
CREATIONDATE,18250
CREATIONTIME,11:12:19
SALESDOCUMENTITEMCATEGORY,TAN
PRODUCT,5629488576
SOLDTOPARTY,3078518136


## Step 3. `explain`: which past orders does the answer rest on?

The function below does three things and prints each one:

- **3a. Candidates:** take the `K` past orders most similar to the new one.
- **3b. Hide and ask:** hide a random half of the candidates, ask TabPFN again, and note how sure it is of the same answer. Repeat `M` times.
- **3c. Score:** work out, for every candidate, how much higher the confidence is when that order is in view. The 10 highest scores are the precedents.

In [9]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors

scaler = StandardScaler().fit(E_ctx)
nn = NearestNeighbors(n_neighbors=min(K, len(E_ctx))).fit(scaler.transform(E_ctx))
ALL = np.arange(len(X_ctx))

def confidence(rows, x_row, answer):
    # How sure is TabPFN of `answer` for x_row, if it sees only these past orders?
    ys_ = y_ctx[rows]
    classes = np.unique(ys_)
    if len(classes) == 1:
        return float(classes[0] == answer), classes[0]
    m = new_model().fit(X_ctx.iloc[rows], ys_)
    p = dict(zip(m.classes_, m.predict_proba(x_row)[0].astype(float)))
    return p.get(answer, 0.0), max(p, key=p.get)

def explain(order, verbose=True):
    x_row, answer = X_tst.iloc[[order]], pred_full[order]
    rng = np.random.default_rng(SEED + order)

    # 3a. candidates
    cand = nn.kneighbors(scaler.transform(E_tst.iloc[[order]]), return_distance=False)[0]
    k = len(cand)
    if verbose:
        counts = pd.Series(y_ctx[cand]).value_counts()
        print(f"3a. {k} candidate past orders. Their payment terms: " + ", ".join(f"'{c}' x{n}" for c, n in counts.items()))

    # 3b. hide and ask
    t0 = time.time()
    masks = np.zeros((M, k), dtype=bool); outs = np.zeros(M)
    for r in range(M):
        masks[r, rng.choice(k, size=k // 2, replace=False)] = True
        outs[r], _ = confidence(cand[masks[r]], x_row, answer)
        if verbose and r < 5:
            print(f"3b. round {r + 1}: shown {masks[r].sum()} of {k} candidates -> {outs[r]:.0%} sure of '{answer}'")
    if verbose:
        print(f"    ... {M} rounds in {time.time() - t0:.0f}s. Confidence ranged from {outs.min():.0%} to {outs.max():.0%}.")
        if outs.std() < 1e-6:
            print("    Note: confidence was the same in every round, so no order stands out. Try another order.")

    # 3c. score every candidate
    scores = Ridge(alpha=1.0).fit(masks.astype(float), outs).coef_
    order_ = np.argsort(-scores)[:TOP_K]
    precedents = cand[order_]
    if verbose:
        print(f"3c. Top-{TOP_K} influence scores: " + ", ".join(f"{s:+.2f}" for s in scores[order_]))
    return {"order": int(order), "answer": answer, "precedent_rows": precedents,
            "influence": scores[order_], "candidates": cand, "rng": rng}

result = explain(ORDER)

3a. 60 candidate past orders. Their payment terms: '32' x15, '33' x13, '03' x10, '01' x9, '00' x7, '70' x4, '96' x1, '42' x1
3b. round 1: shown 30 of 60 candidates -> 18% sure of '03'
3b. round 2: shown 30 of 60 candidates -> 89% sure of '03'
3b. round 3: shown 30 of 60 candidates -> 19% sure of '03'
3b. round 4: shown 30 of 60 candidates -> 29% sure of '03'
3b. round 5: shown 30 of 60 candidates -> 94% sure of '03'
    ... 180 rounds in 74s. Confidence ranged from 5% to 99%.
3c. Top-10 influence scores: +0.49, +0.04, +0.04, +0.04, +0.03, +0.03, +0.03, +0.03, +0.03, +0.02


## Step 4. Read the explanation

The table puts the new order next to its precedents. A tick means the precedent has the **same value** as the new order in that field. Look for precedents that share the customer and already carry the model's answer.

In [10]:
def show_explanation(result):
    order, rows = result["order"], result["precedent_rows"]
    new = X_tst.iloc[order]
    key_fields = [c for c in ["SALESORGANIZATION", "BILLINGCOMPANYCODE", "TRANSACTIONCURRENCY", "SALESDOCUMENTTYPE",
                              "SOLDTOPARTY", "PAYERPARTY", "SHIPTOPARTY", "PRODUCT"] if c in X_ctx.columns]
    table = pd.DataFrame({"past order": rows, "influence": np.round(result["influence"], 3),
                          "its payment terms": y_ctx[rows]})
    table["same answer?"] = np.where(table["its payment terms"] == result["answer"], "yes", "no")
    for c in key_fields:
        vals = X_ctx.iloc[rows][c].to_numpy()
        table[c] = [f"{v}  ✓" if v == new[c] else str(v) for v in vals]
    table["fields matching"] = [int(sum(X_ctx.iloc[r][c] == new[c] for c in key_fields)) for r in rows]
    table.index = [f"precedent {n + 1}" for n in range(len(rows))]
    print(f"New order {order}: model says '{result['answer']}'. Its key fields:")
    print("  " + " | ".join(f"{c}={new[c]}" for c in key_fields))
    return table

show_explanation(result)

New order 7: model says '03'. Its key fields:
  SALESORGANIZATION=0010 | BILLINGCOMPANYCODE=D011 | TRANSACTIONCURRENCY=EUR | SALESDOCUMENTTYPE=TA | SOLDTOPARTY=3078518136 | PAYERPARTY=9946326346 | SHIPTOPARTY=8708984665 | PRODUCT=5629488576


,past order,influence,its payment terms,same answer?,SALESORGANIZATION,BILLINGCOMPANYCODE,TRANSACTIONCURRENCY,SALESDOCUMENTTYPE,SOLDTOPARTY,PAYERPARTY,SHIPTOPARTY,PRODUCT,fields matching
precedent 1,1329,0.491,03,yes,0010 ✓,D011 ✓,EUR ✓,TA ✓,3078518136 ✓,9946326346 ✓,8708984665 ✓,3641914149,7
precedent 2,893,0.042,33,no,0010 ✓,D011 ✓,EUR ✓,ZMUN,8383348969,8383348969,8383348969,0965030702,3
precedent 3,1170,0.039,03,yes,0010 ✓,D011 ✓,EUR ✓,ZMUN,6710885519,6710885519,6710885519,1330408554,3
precedent 4,1127,0.039,00,no,0010 ✓,D011 ✓,EUR ✓,ZMUN,7481075268,7481075268,7481075268,0422017490,3
precedent 5,373,0.034,32,no,0010 ✓,D011 ✓,EUR ✓,TA ✓,4742731422,4742731422,4742731422,1190498210,4
precedent 6,1981,0.030,32,no,0010 ✓,D011 ✓,EUR ✓,TA ✓,4742731422,4742731422,4742731422,2871001594,4
precedent 7,1756,0.030,33,no,0300,NL30,EUR ✓,ZIA,8383348969,8383348969,8383348969,2791103910,1
precedent 8,664,0.030,00,no,0010 ✓,D011 ✓,EUR ✓,ZMUN,7481075268,7481075268,7481075268,4202171294,3
precedent 9,156,0.029,33,no,0010 ✓,D011 ✓,EUR ✓,ZMUN,7203485562,7203485562,7203485562,2104588456,3
precedent 10,98,0.024,00,no,0010 ✓,D011 ✓,EUR ✓,ZMUN,7481075268,7481075268,7481075268,0422017490,3


## Step 5. `verify`: does the answer really rest on those orders?

Remove the 10 precedents from **all 2,000** past orders and ask the full model again. For comparison, remove 10 random candidates instead, three times.

The explanation is **verified** if removing the precedents lowers confidence by at least 5 points and by more than every random removal.

In [11]:
def verify(result, verbose=True):
    order, answer = result["order"], result["answer"]
    x_row, rng, cand = X_tst.iloc[[order]], result["rng"], result["candidates"]
    p_before = float(conf_full[order])

    p_after, new_answer = confidence(np.setdiff1d(ALL, result["precedent_rows"]), x_row, answer)
    
    # Random comparison: 10 candidates that are NOT precedents
    others = np.setdiff1d(cand, result["precedent_rows"])
    random_drops = []
    for _ in range(N_RANDOM):
        p_rand, _ = confidence(np.setdiff1d(ALL, rng.choice(others, size=TOP_K, replace=False)), x_row, answer)
        random_drops.append(p_before - p_rand)
    
    drop = p_before - p_after
    verified = bool(drop >= 0.05 and drop > max(random_drops))
    if verbose:
        print(f"Before removal            : {p_before:.0%} sure of '{answer}'")
        print(f"Precedents removed        : {p_after:.0%} sure of '{answer}'  (fall of {drop * 100:.0f} points)")
        print(f"Model's answer afterwards : '{new_answer}'" + ("  <- the answer changed" if new_answer != answer else "  (unchanged)"))
        print("10 random orders removed  : falls of " + ", ".join(f"{d * 100:.0f}" for d in random_drops) + " points")
        print("\nVERDICT:", "VERIFIED - the answer does rest on these precedents." if verified
              else "NOT VERIFIED - removing these orders did not move the full model enough.")
    return {"confidence_before": round(p_before, 4), "confidence_after": round(p_after, 4),
            "fall_points": round(drop * 100, 1), "answer_after": new_answer,
            "answer_changed": bool(new_answer != answer),
            "random_fall_points": [round(d * 100, 1) for d in random_drops], "verified": verified}

check = verify(result)

Before removal            : 97% sure of '03'
Precedents removed        : 21% sure of '03'  (fall of 77 points)
Model's answer afterwards : '01'  <- the answer changed
10 random orders removed  : falls of 0, 1, 0 points

VERDICT: VERIFIED - the answer does rest on these precedents.


## Step 6. The final result

This is what the tool would hand back for one order, and what would be written to the audit ledger: the prediction, the precedents, and whether the explanation passed the check.

In [12]:
record = {
    "order": result["order"],
    "predicted_payment_terms": str(result["answer"]),
    "true_payment_terms": str(y_tst[result["order"]]),
    "precedents": [{"past_order": int(r), "influence": round(float(s), 3), "payment_terms": str(y_ctx[r]),
                    "same_customer": bool(X_ctx.iloc[r][CUSTOMER_COL] == X_tst.iloc[result["order"]][CUSTOMER_COL])}
                   for r, s in zip(result["precedent_rows"], result["influence"])],
    "verification": check,
}
print(json.dumps(record, indent=2))

{
  "order": 7,
  "predicted_payment_terms": "03",
  "true_payment_terms": "03",
  "precedents": [
    {
      "past_order": 1329,
      "influence": 0.491,
      "payment_terms": "03",
      "same_customer": true
    },
    {
      "past_order": 893,
      "influence": 0.042,
      "payment_terms": "33",
      "same_customer": false
    },
    {
      "past_order": 1170,
      "influence": 0.039,
      "payment_terms": "03",
      "same_customer": false
    },
    {
      "past_order": 1127,
      "influence": 0.039,
      "payment_terms": "00",
      "same_customer": false
    },
    {
      "past_order": 373,
      "influence": 0.034,
      "payment_terms": "32",
      "same_customer": false
    },
    {
      "past_order": 1981,
      "influence": 0.03,
      "payment_terms": "32",
      "same_customer": false
    },
    {
      "past_order": 1756,
      "influence": 0.03,
      "payment_terms": "33",
      "same_customer": false
    },
    {
      "past_order": 664,
      "influen

## Try another order

Change the number and run the cell. Each order takes about 1.5 minutes. Try a few: you should see some that are verified and some that are not, as in run 3.

In [13]:
OTHER = 5      # any number from 0 to 999

r2 = explain(OTHER)
display(show_explanation(r2))
c2 = verify(r2)

3a. 60 candidate past orders. Their payment terms: '32' x18, '33' x13, '03' x12, '00' x9, '01' x3, '70' x2, '95' x2, '42' x1
3b. round 1: shown 30 of 60 candidates -> 11% sure of '33'
3b. round 2: shown 30 of 60 candidates -> 15% sure of '33'
3b. round 3: shown 30 of 60 candidates -> 24% sure of '33'
3b. round 4: shown 30 of 60 candidates -> 13% sure of '33'
3b. round 5: shown 30 of 60 candidates -> 15% sure of '33'
    ... 180 rounds in 73s. Confidence ranged from 7% to 53%.
3c. Top-10 influence scores: +0.03, +0.03, +0.02, +0.02, +0.02, +0.01, +0.01, +0.01, +0.01, +0.01
New order 5: model says '33'. Its key fields:
  SALESORGANIZATION=0700 | BILLINGCOMPANYCODE=PL10 | TRANSACTIONCURRENCY=EUR | SALESDOCUMENTTYPE=ZMUN | SOLDTOPARTY=8895508015 | PAYERPARTY=8895508015 | SHIPTOPARTY=0851548071 | PRODUCT=1546872367


,past order,influence,its payment terms,same answer?,SALESORGANIZATION,BILLINGCOMPANYCODE,TRANSACTIONCURRENCY,SALESDOCUMENTTYPE,SOLDTOPARTY,PAYERPARTY,SHIPTOPARTY,PRODUCT,fields matching
precedent 1,687,0.027,33,yes,0300,NL30,EUR ✓,TA,2933266658,2933266658,2933266658,3268663389,1
precedent 2,1764,0.026,33,yes,0700 ✓,PL10 ✓,EUR ✓,ZMUN ✓,0212677443,0212677443,0212677443,3755243785,4
precedent 3,1646,0.024,33,yes,0800,RO10,EUR ✓,ZMUN ✓,2139942130,2139942130,2139942130,3709189839,2
precedent 4,1690,0.019,33,yes,0700 ✓,PL10 ✓,EUR ✓,ZMUN ✓,0212677443,0212677443,0212677443,8298592021,4
precedent 5,454,0.018,33,yes,0300,NL30,EUR ✓,ZMUN ✓,2943589161,2943589161,2943589161,6376558383,2
precedent 6,782,0.015,33,yes,0700 ✓,PL10 ✓,EUR ✓,ZMUN ✓,0212677443,0212677443,0212677443,4803449119,4
precedent 7,958,0.014,33,yes,0700 ✓,PL10 ✓,EUR ✓,ZMUN ✓,0212677443,0212677443,0212677443,4913252217,4
precedent 8,1831,0.013,03,no,0010,D011,EUR ✓,ZMUN ✓,2972983400,2972983400,2972983400,3755243785,2
precedent 9,1938,0.012,33,yes,0700 ✓,PL10 ✓,EUR ✓,ZMUN ✓,1974833849,1974833849,1974833849,2250180452,4
precedent 10,508,0.012,00,no,0010,D011,EUR ✓,ZMUN ✓,1946124332,1946124332,7143172224,0422017490,2


Before removal            : 24% sure of '33'
Precedents removed        : 17% sure of '33'  (fall of 7 points)
Model's answer afterwards : '32'  <- the answer changed
10 random orders removed  : falls of -1, 2, 0 points

VERDICT: VERIFIED - the answer does rest on these precedents.
